# LexChain LLM analysis benchmark — self-hostable candidates vs 70B reference

**Pivot:** LexChain must run its analysis model on its own infrastructure, so we select among **4 self-hostable Ollama candidates** (4 different families); the NIM-hosted 70B is kept **only as a reference ceiling** and is labeled *reference (non-deployable)* everywhere.

| Model | Backend | Role | Context used |
|---|---|---|---|
| `llama3.1:8b` | Ollama | candidate (Meta) | 32,768 |
| `qwen3:14b` | Ollama | candidate (Qwen; thinking disabled, `<think>` stripped) | 32,768 |
| `mistral-nemo:12b` | Ollama | candidate (Mistral; 128k native ctx) | 32,768 |
| `gemma3:27b` | Ollama | candidate (Google) ⚠ ~17 GB VRAM — **L4/A100 runtime required**; the runner hard-asserts free VRAM before calling it | 32,768 |
| `meta/llama-3.1-70b-instruct` | NVIDIA NIM | **reference (non-deployable)** | hosted |

**Excluded at eligibility** (recorded in `models_meta.json`): `phi4:14b` — 16k context cannot hold 2/10 sampled documents without truncation; candidates must process every document whole.

One **frozen prompt** (v2.0-cuad-checklist: CUAD-derived 12-category risk checklist, present/absent + verbatim quote), temperature 0, identical JSON schema for all 5.

**GUARDRAIL:** the ground-truth key must be hand-authored **before** any model output is rendered. The notebook is ordered accordingly and `build_blind_eval` refuses to run until `ground_truth_key.csv` is filled.

Set Colab secret `NVIDIA_API_KEY` (for the reference model) before starting.

In [ ]:
# Cell 1 — mount Drive, clone repo, install deps, load NIM key
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
if not os.path.exists('/content/lexchain-chunk-embed-bench'):
    !git clone https://github.com/MarvinPescos/lexchain-chunk-embed-bench.git /content/lexchain-chunk-embed-bench
%cd /content/lexchain-chunk-embed-bench
!git pull
!pip install -q -r requirements-analysis.txt
os.environ['NVIDIA_API_KEY'] = userdata.get('NVIDIA_API_KEY')
print('NIM key loaded:', bool(os.environ.get('NVIDIA_API_KEY')))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Cell 2 — install Ollama, launch it (32k ctx), pull the 4 candidates (~25 GB total)
# gemma3:27b needs ~17 GB VRAM: use an L4/A100 runtime. The runner also
# hard-asserts free VRAM before any gemma3:27b call (no silent CPU offload).
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, os, time
env = {**os.environ, 'OLLAMA_CONTEXT_LENGTH': '32768'}
subprocess.Popen(['ollama', 'serve'], env=env,
                 stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
for m in ['llama3.1:8b', 'qwen3:14b', 'mistral-nemo:12b', 'gemma3:27b']:
    !ollama pull {m}
!ollama list
# exact tag + quantization for the paper (also recorded to models_meta.json by the runner)
for m in ['llama3.1:8b', 'qwen3:14b', 'mistral-nemo:12b', 'gemma3:27b']:
    print('---', m); !ollama show {m} | head -12

In [ ]:
# Cell 3 — data + ground-truth authoring materials + CONTEXT SAFETY CHECK
# (no model output is produced or shown here)
!python download_data.py
!python -m analysis.prepare_ground_truth

import sys; sys.path.insert(0, '.')
from analysis.data import load_docs, sample_stems, context_check
docs = load_docs(); stems = sample_stems(docs.keys(), n=10)
report = context_check({s: docs[s] for s in stems})  # raises loudly on any misfit
print('CONTEXT CHECK PASSED — all 10 docs fit all models.')
print('prompt overhead:', report['prompt_overhead_tokens'], 'tokens')
for m, info in report['per_model'].items():
    print(f"  {m}: worst case {info['worst_case_tokens']:,} / limit {info['limit']:,}")

## ✋ GT-AUTHORING GATE — do this before running any model

1. Download `doc_texts/*.txt` + `ground_truth_key_template.csv` from `MyDrive/lexchain_bench/analysis/`.
2. Hand-author the key from the documents (never from model outputs — they don't exist yet).
3. Save it as `ground_truth_key.csv` in the same Drive folder **and commit it** to your records.

Cells 4–5 only produce checkpoints (status/latency logs, no content shown). Cell 6 — the first cell that *renders* outputs — refuses to run until the key is in place.

In [ ]:
# Cell 4 — smoke: 2 docs x 5 models (10 calls); logs show status/latency only
!python -m analysis.analyze --smoke 2

import json
from pathlib import Path
cache = Path('/content/drive/MyDrive/lexchain_bench/analysis')
recs = [json.loads(p.read_text()) for p in (cache/'analyses').glob('*.json')]
ok = sum(r.get('ok', False) for r in recs)
print(f"{ok}/{len(recs)} outputs parsed + schema-valid")
by_model = {}
for r in recs:
    by_model.setdefault(r['model'], []).append(r.get('latency_s') or 0)
proj = 0
for m, ls in sorted(by_model.items()):
    mean = sum(ls)/len(ls); proj += mean*10
    print(f"  {m}: {mean:.1f}s/doc ({r['latency_label']}) -> ~{mean*10/60:.1f} min for 10 docs")
print(f"Projected full run (50 calls, sequential): ~{proj/60:.0f} min")

In [ ]:
# Cell 5 — full run: 10 docs x 5 models = 50 outputs (resumable; reruns skip)
!python -m analysis.analyze

In [ ]:
# Cell 6 — blind sheet (first output-rendering step; guardrail-enforced)
!python -m analysis.build_blind_eval
print('\nblind_eval.csv: 50 rows, Output A–E labels, SummEval columns')
print('Rate coherence / consistency / fluency / relevance (1–5) + hallucination notes.')
print('Do NOT open unblinding_key.csv until all rating is done.')

### Human step (off-Colab)
Rate all 50 rows of `blind_eval.csv`. The four dimensions follow **SummEval**: coherence (organization), consistency (factual alignment with the source), fluency (grammar/readability), relevance (captures important content). Then run Cell 7.

In [ ]:
# Cell 7 — aggregate: un-blind, entity/risk F1 vs your key, wins, final table
!python -m analysis.aggregate_analysis